# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r ndvi_da gila_boundary_gdf ndvi_pre ndvi_post ndvi_diff

no stored variable or alias ndvi_diff


You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries
### working with dataframes
import pandas as pd

### working with geodataframes
import geopandas as gpd

### making interactive plots of tabular and vector data
import hvplot.pandas

### work with json data
import json

### work with pattern matching rules for pathnames 
from glob import glob

### access the data we'll use for the assignment
import earthpy

### interactive plots with rasters 
import hvplot.xarray

### handle geospatial rasters
import rioxarray as rxr

### work with arrays
import xarray as xr

## Step 5: Is the mean annual July NDVI different within the Gila River Indian Community after the change in water rights?

You will compute the mean NDVI inside and outside the Reservation boundary. 

### Step 5A: Get the NDVI inside and outside the Gila River Indian Community

First, use the code below to get a `GeoDataFrame` of the area outside the Reservation.

- Check the variable names. Make sure that the code uses your boundary `GeoDataFrame`  
- How could you test if the geometry was modified correctly? Add some code to take a look at the results

In [3]:
### compute the area outside the reservation boundary
out_gdf = (

    ### make boundary box around Gila boundary, helps deal with non-straight lines
    gpd.GeoDataFrame(geometry = gila_boundary_gdf.envelope)

    ### subtract the Gila boundary from the bounding box
    .overlay(gila_boundary_gdf, how = 'difference'))

out_gdf

,geometry
0,"MULTIPOLYGON (((-112.30875 32.96704, -112.3087..."


Next, clip your DataArray to the boundaries for both inside and outside
the reservation. You will need to replace the `GeoDataFrame` name with
your own. Check out the [lesson on clipping data with the `rioxarray`
library on EarthDataScience.org](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/crop-raster-data-with-shapefile-in-python/).

**Note**: It’s important to use `from_disk = True` when clipping large arrays like this. It allows the computer to use less valuable memory resources when clipping. Otherwise, you will probably find that the cell below crashes your kernel.

In [4]:
### clip data to inside the reservation's boundaries
ndvi_in_da = ndvi_da.rio.clip(gila_boundary_gdf.geometry, from_disk = True)

### clip data to outside the reservation's boundaries 
ndvi_out_da = ndvi_da.rio.clip(out_gdf.geometry, from_disk = True)

ndvi_out_da

<xarray.Dataset> Size: 48MB
Dimensions:      (x: 380, y: 203, date: 154)
Coordinates:
  * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
  * y            (y) float64 2kB 33.39 33.39 33.38 33.38 ... 32.97 32.97 32.97
  * date         (date) datetime64[us] 1kB 2001-01-14 2001-01-16 ... 2022-01-24
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (date, y, x) float32 48MB 0.6146 0.3796 ... 0.1736 0.2146

### Step 5B: Assess the change in NDVI before and after the change in water rights, inside and outside the boundaries of the Gila River Indian Community

For **both inside and outside** the Gila River Indian Community boundary, calculate the mean annual July NDVI:  

- Group the data by year  
- Take the mean. You always need to tell reducing methods in `xarray` what dimensions you want to reduce. When you want to summarize data across **all** dimensions, you can use the `...` syntax, e.g. `.mean(...)` as a shorthand  
- Select the NDVI variable  
- Convert to a DataFrame using the `to_dataframe()` method  
- Join the two DataFrames for plotting using the `.join()` method. You will need to rename the columns using the `lsuffix =` and `rsuffix =` parameters
- Finally, plot annual July means for both inside and outside the Reservation on the same plot.

**Note**: The DateIndex in pandas is a little different from the Datetime Dimension in xarray. You will need to use the `.dt.year` syntax to access information about the year, not just `.year`.

In [5]:
### calculate mean annual July NDVI inside
july_ndvi_in_df = (
ndvi_in_da

    ### group by year
    .groupby(ndvi_in_da.date.dt.year)

    ### take the mean of all the values in each group
    .mean(...)

    ### extract NDVI and put it in a dataframe
    .NDVI.to_dataframe()
)

# ### to see the july NDVI for inside the Gila boundary
# july_ndvi_in_df

### calculate mean annual July NDVI outside
july_ndvi_out_df = (
ndvi_out_da

    ### group by year
    .groupby(ndvi_out_da.date.dt.year)

    ### take the mean of all the values in each group
    .mean(...)

    ### extract NDVI and put it in a dataframe
    .NDVI.to_dataframe()
)
july_ndvi_out_df

### plot inside and outside the reservation on a single plot


### make it interactive

,band,spatial_ref,NDVI
year,,,
2001,1,0,0.247629
2002,1,0,0.226726
2003,1,0,0.229889
2004,1,0,0.221753
2005,1,0,0.255275
2006,1,0,0.235571
2007,1,0,0.211984
2008,1,0,0.237739
2009,1,0,0.218907


In [6]:
### make dataframe with in and out values
july_ndvi_df = (

    ### grab NDVI from inside
    july_ndvi_in_df[['NDVI']]

    ### join it to the out_df
    .join(

        ### grab NDVI from outside GRIC
        july_ndvi_out_df[['NDVI']],

        ### make column names for NDVI distinct and descriptive
        lsuffix = ' inside GRIC', rsuffix = ' outside GRIC'
    )
)
july_ndvi_df

/var/folders/rf/kx7zk_n141s5pf1f5k_8d3qw0000gn/T/ipykernel_23328/2511525436.py:8: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  .join(


,NDVI inside GRIC,NDVI outside GRIC
year,,
2001,0.199645,0.247629
2002,0.177933,0.226726
2003,0.187301,0.229889
2004,0.176162,0.221753
2005,0.238630,0.255275
2006,0.211491,0.235571
2007,0.181710,0.211984
2008,0.201902,0.237739
2009,0.179118,0.218907


Now, take the difference between outside and inside the site boundary and plot that


In [7]:
### calculate difference in mean July NDVI
july_ndvi_df['difference'] = (
    july_ndvi_df['NDVI inside GRIC']
    - july_ndvi_df['NDVI outside GRIC']
)

### add bokeh extension
hvplot.extension('bokeh')

### plot difference inside and outside the boundary
july_ndvi_df.difference.hvplot(
    title = 'Difference between NDVI Within and Outside GRIC over Time'
)

:Curve   [year]   (difference)

In [9]:
### save html of plot
export_july_ndvi = (july_ndvi_df.difference.hvplot(
    title = 'Difference Between NDVI Within and Outside GRIC Over Time')
)

hvplot.save(export_july_ndvi, 'NDVI_difference_GRIC_border.html')

In [12]:
### calculate the mean NDVI per pixel per year
ndvi_mean_all = (

    ### take the NDVI raster stack
    ndvi_da

          ### group by year
            .groupby(ndvi_in_da.date.dt.year)

           ### take the mean of all the values in each group
              .mean('date')
)
ndvi_mean_all

<xarray.Dataset> Size: 7MB
Dimensions:      (year: 22, y: 203, x: 382)
Coordinates:
  * year         (year) int64 176B 2001 2002 2003 2004 ... 2019 2020 2021 2022
  * y            (y) float64 2kB 33.39 33.39 33.38 33.38 ... 32.97 32.97 32.97
  * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (year, y, x) float32 7MB 0.634 0.4888 0.4488 ... 0.1966 0.2166

In [13]:
### plot the insde and outside data arrays to explore where the differences are in the mean annual peak NDVI (peak growing season)
(
   ndvi_mean_all.hvplot(
    
    ### tell it which coordinates are lat/long
    x = 'x', y = 'y', 
    
    ### give it a color ramp
    cmap = 'YlGn', 
    
    ### add basemap CHANGE BACK TO TRUE WHEN YOU FIGURE OUT TRACEBACK ERROR
    geo = True)
        
    *

    ### add the Gila River Indian Community boundary to your map
    gila_boundary_gdf.hvplot(geo = True, fill_color = None, line_color = 'black')
)

/opt/miniconda3/envs/earth-analytics-python-lj/lib/python3.11/site-packages/shapely/creation.py:746: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/opt/miniconda3/envs/earth-analytics-python-lj/lib/python3.11/site-packages/shapely/creation.py:746: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/opt/miniconda3/envs/earth-analytics-python-lj/lib/python3.11/site-packages/shapely/creation.py:746: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/opt/miniconda3/envs/earth-analytics-python-lj/lib/python3.11/site-packages/shapely/creation.py:746: RuntimeWarning: invalid value encountered in create_collection
  return lib.create_collection(geometries, np.intc(typ), out=out, **kwargs)
/opt/miniconda3/envs/earth-analytics-pyt

BokehModel(combine_events=True, render_bundle={'docs_json': {'6d39f3dc-3beb-4f34-97b6-0e544ae79d5e': {'version…

In [ ]:
### create plot showing how each NDVI annual average changes over time
july_ndvi_df.hvplot.line(

    ### plot both annual averages in same plot
    y = ['NDVI inside GRIC', 'NDVI outside GRIC'],

    ### give the y axis a label
    value_label = 'Average Annual NDVI',

    ### give the plot a title
    title = 'Comparison of NDVI Inside and Outside of GRIC'
)

:NdOverlay   [Variable]
   :Curve   [year]   (Average Annual NDVI)

### Reflect and Respond #6:
What do you observe? Don’t forget to write a headline and description of your plot!

### NDVI within the GIla River Indian Community Increasing Compared to Surrounding Areas

The plot titled ‘Difference Between NDVI Within and Outside GRIC Over Time’ covers the years 2001 to 2022, with the water rights settlement occurring in 2004. All values on the plot are negative, indicating that the average annual NDVI outside the GRIC boundary is always more than inside. Some of the greatest differences in NDVI occurred between 2001 and 2004, though in 2005 the difference in NDVI was significantly less. There is an overall increase in the difference after 2005 until 2011. After 2011, there is an overall decrease in the difference, potentially indicating that the restoration of water rights has helped increase the average annual NDVI within GRIC, though there could be other factors driving this change.

The completion of infrastructure projects and new water management practices since the 2004 decision have most likely already restored some of the agriculture and green areas in GRIC. Rising temperatures and drought conditions over the last few years stressing vegetation along the Gila River (Hartfield et al., 2020), could help explain why NDVI within GRIC has not quite caught up to the surrounding areas. Furthermore, when looking at the map exploring the difference in NDVI after water rights were restored, there are many areas that experience positive and negative change both within and outside the borders. The community is south of Phoenix, which is experiencing rapid urbanization, butting up against natural and political barriers, including reservations (Shrestha et al., 2012). The overall difference in NDVI could be driven by a decrease in agriculture, and thus NDVI, outside of the GRIC boundaries, due to urbanization. 

### References

Hartfield, K., Leeuwen, W. J. D. V., & Gillan, J. K. (2020). Remotely Sensed Changes in Vegetation Cover Distribution and Groundwater along the Lower Gila River. *Land, 9*(9), 326. [https://doi.org/10.3390/land9090326](https://doi.org/10.3390/land9090326)

Shrestha, M. K., York, A. M., Boone, C. G., & Zhang, S. (2012). Land fragmentation due to rapid urbanization in the Phoenix Metropolitan Area: Analyzing the spatiotemporal patterns and drivers. *Applied Geography, 32*(2), 522–531. [https://doi.org/10.1016/j.apgeog.2011.04.004](https://doi.org/10.1016/j.apgeog.2011.04.004)


## Wrap up

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!